In [2]:
import os 
os.chdir('../')
%pwd

'c:\\Users\\DELL\\Desktop\\drink_quality_prediction'

In [11]:
from pathlib import Path
from dataclasses import dataclass
from typing import Dict

@dataclass(frozen=True)
class ModelEvaluationConfig:
    
    root_dir: Path
    
    X_test_path: Path
    y_test_path: Path
    
    model_path: Path
    
    params: Dict
    
    metric_file_name: Path   # ✅ ADD THIS
    
    target_column: str       # (optional if you use it)

In [12]:
# configManager:
from mlproject.utils.common import read_yaml, create_directories
from mlproject.constants import * 


In [13]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        
        config = self.config.model_evaluation
        params = self.params.random_forest
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])
        
        model_evaluation_config = ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            X_test_path=Path(config.X_test_path),
            y_test_path=Path(config.y_test_path),
            model_path=Path(config.model_path),
            params=params,
            metric_file_name=Path(config.metric_file_name),  # ✅ must match
            target_column=schema
        )
        
        return model_evaluation_config

In [14]:
#components

from mlproject import logger
import os 
import pandas 
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import numpy 
import joblib



In [15]:
from mlproject.utils.common import save_json



In [16]:
import pandas as pd
import joblib
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)




class ModelEvaluation:
    
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config


    # ==========================================
    # 1️⃣ Separate Metrics Function
    # ==========================================
    def calculate_metrics(self, y_true, y_pred, y_prob):
        
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        roc_auc = roc_auc_score(y_true, y_prob)
        conf_matrix = confusion_matrix(y_true, y_pred)

        metrics = {
            "accuracy": round(accuracy, 4),
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1_score": round(f1, 4),
            "roc_auc": round(roc_auc, 4),
            "confusion_matrix": conf_matrix.tolist()
        }

        return metrics


    # ==========================================
    # 2️⃣ Main Evaluate Function
    # ==========================================
    def evaluate(self):

        # -----------------------------
        # Load Test Data
        # -----------------------------
        X_test = pd.read_csv(self.config.X_test_path)
        y_test = pd.read_csv(self.config.y_test_path).values.ravel()

        logger.info("Test data loaded successfully.")

        # -----------------------------
        # Load Model
        # -----------------------------
        model = joblib.load(self.config.model_path)

        logger.info("Trained model loaded successfully.")

        # -----------------------------
        # Prediction
        # -----------------------------
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]  # binary prob

        logger.info("Prediction completed.")

        # -----------------------------
        # Calculate Metrics (Separate FN)
        # -----------------------------
        metrics = self.calculate_metrics(y_test, y_pred, y_prob)

        print("\n📊 Binary Classification Metrics:\n")
        for key, value in metrics.items():
            print(f"{key}: {value}")

        print("\nClassification Report:\n")
        print(classification_report(y_test, y_pred))

        # -----------------------------
        # Save Metrics JSON
        # -----------------------------
        save_json(
            path=self.config.metric_file_name,
            data=metrics
        )

        logger.info("Metrics saved as JSON successfully.")

        return metrics

In [17]:
try:
    config = ConfigurationManager()
    Model_evaluation_config=config.get_model_evaluation_config()
    model_eval=ModelEvaluation(Model_evaluation_config)
    model_eval.evaluate()
    
except Exception as e :
    raise e

[2026-03-17 01:53:30,944] INFO: common: yaml file: config\config.yaml loaded successfully
[2026-03-17 01:53:30,945] INFO: common: yaml file: params.yaml loaded successfully
[2026-03-17 01:53:30,947] INFO: common: yaml file: schema.yaml loaded successfully
[2026-03-17 01:53:30,948] INFO: common: created directory at: artifacts
[2026-03-17 01:53:30,948] INFO: common: created directory at: artifacts/model_evaluation
[2026-03-17 01:53:30,969] INFO: 1575121255: Test data loaded successfully.
[2026-03-17 01:53:31,049] INFO: 1575121255: Trained model loaded successfully.
[2026-03-17 01:53:31,106] INFO: 1575121255: Prediction completed.

📊 Binary Classification Metrics:

accuracy: 0.7721
precision: 0.8324
recall: 0.713
f1_score: 0.7681
roc_auc: 0.8432
confusion_matrix: [[161, 31], [62, 154]]

Classification Report:

              precision    recall  f1-score   support

           0       0.72      0.84      0.78       192
           1       0.83      0.71      0.77       216

    accuracy    